In [17]:
import torch
import pandas as pd
import torch_geometric
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import RandomLinkSplit

In [18]:
df = pd.read_csv('../data/edges/clean_triples.csv')

In [22]:
def build_type_maps(df):
    maps = {}
    for t in pd.concat([df["type_entity_1"], df["type_entity_2"]]).unique():
        ids = pd.concat([
            df.loc[df["type_entity_1"] == t, "id_entity_1"],
            df.loc[df["type_entity_2"] == t, "id_entity_2"],
        ]).unique()
        maps[t] = {int(rid): i for i, rid in enumerate(ids)}
    return maps

def init_nodes(data, maps, emb_dim=64, device="cpu"):
    for node_type, m in maps.items():
        n = len(m)
        data[node_type].x = torch.randn(n, emb_dim, device=device)
    return data


def add_edges(data, df, maps, device="cpu", make_reverse=False):
    # сгруппируем по (type1, pred, type2)
    for (t1, pred, t2), g in df.groupby(["type_entity_1", "predicate", "type_entity_2"], sort=False):
        src = torch.tensor([maps[t1][int(x)] for x in g["id_entity_1"].tolist()],
                           dtype=torch.long, device=device)
        dst = torch.tensor([maps[t2][int(x)] for x in g["id_entity_2"].tolist()],
                           dtype=torch.long, device=device)
        ei = torch.stack([src, dst], dim=0)  # [2, E]
        data[(t1, pred, t2)].edge_index = ei

        if make_reverse:
            data[(t2, f"{pred}", t1)].edge_index = torch.stack([dst, src], dim=0)
    return data


def df_to_heterodata(df, emb_dim=64, device="cpu", make_reverse=False):
    df = df.copy()
    maps = build_type_maps(df)

    data = HeteroData()
    data = init_nodes(data, maps, emb_dim=emb_dim, device=device)
    data = add_edges(data, df, maps, device=device, make_reverse=make_reverse)

    return data, maps

data, maps = df_to_heterodata(df, emb_dim=64, make_reverse=True, device="cuda")
data

HeteroData(
  RNA={ x=[738, 64] },
  DNA={ x=[596, 64] },
  NucleicMixed={ x=[23, 64] },
  NucleicAmbigous={ x=[16, 64] },
  AA={ x=[70937, 64] },
  SmallMolecule={ x=[1001342, 64] },
  (RNA, interacts_with, AA)={ edge_index=[2, 340] },
  (AA, interacts_with, RNA)={ edge_index=[2, 340] },
  (DNA, interacts_with, AA)={ edge_index=[2, 323] },
  (AA, interacts_with, DNA)={ edge_index=[2, 323] },
  (NucleicMixed, interacts_with, AA)={ edge_index=[2, 10] },
  (AA, interacts_with, NucleicMixed)={ edge_index=[2, 10] },
  (NucleicAmbigous, interacts_with, AA)={ edge_index=[2, 2] },
  (AA, interacts_with, NucleicAmbigous)={ edge_index=[2, 2] },
  (DNA, interacts_with, SmallMolecule)={ edge_index=[2, 311] },
  (SmallMolecule, interacts_with, DNA)={ edge_index=[2, 311] },
  (RNA, interacts_with, SmallMolecule)={ edge_index=[2, 1461] },
  (SmallMolecule, interacts_with, RNA)={ edge_index=[2, 1461] },
  (NucleicAmbigous, interacts_with, SmallMolecule)={ edge_index=[2, 35] },
  (SmallMolecule, inter

In [32]:


def df_to_homo_two_rel(df: pd.DataFrame, emb_dim=64, device="cpu"):
    # 1) глобальные id для узлов: различаем по (entity_type, raw_id)
    #    (если ты хочешь 1 тип узлов, но НЕ сливать разные типы с одинаковым числом)
    nodes = pd.concat([
        df[["type_entity_1","id_entity_1"]].rename(columns={"type_entity_1":"t","id_entity_1":"id"}),
        df[["type_entity_2","id_entity_2"]].rename(columns={"type_entity_2":"t","id_entity_2":"id"}),
    ], axis=0).drop_duplicates()

    key = list(zip(nodes["t"].astype(str), nodes["id"].astype(int)))
    node_map = {k:i for i,k in enumerate(key)}
    num_nodes = len(node_map)

    # 2) edge_index
    src = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_1"], df["id_entity_1"])]
    dst = [node_map[(str(t), int(i))] for t,i in zip(df["type_entity_2"], df["id_entity_2"])]
    edge_index = torch.tensor([src, dst], dtype=torch.long, device=device)

    # 3) edge_type (2 предиката -> 0/1)
    preds = df["predicate"].astype(str).unique().tolist()
    assert len(preds) == 2, f"ожидал 2 predicate, got {preds}"
    pred2id = {p:i for i,p in enumerate(sorted(preds))}
    edge_type = torch.tensor([pred2id[p] for p in df["predicate"].astype(str)], dtype=torch.long, device=device)

    # 4) node features (рандом)
    x = torch.randn(num_nodes, emb_dim, device=device)

    data = Data(x=x, edge_index=edge_index)
    data.edge_type = edge_type  # [E]
    return data, node_map, pred2id

data, node_map, pre2id = df_to_homo_two_rel(df, emb_dim=64, device="cpu")

In [36]:
data.edge_type.unique()

tensor([0, 1])